# 03 — Gold: dim_financial_segment

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_financial_segment` |
| **Grain** | One row per GlobalFinancialSegmentId |
| **Source** | `ref.FinancialSegmentHierarchy` |
| **PK** | `GlobalFinancialSegmentId` (int) |
| **Rows** | 3,993 |

**Hierarchy**: Segment (8) → Business (57) → LOB (177) → ProductService (552) → Team (3,203)

**Cross-sell**: Segment level (CRB/HCB/IRR) is the main cross-sell axis.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_financial_segment"
SOURCE_TABLE = "ref.FinancialSegmentHierarchy"

print(f"✅ Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"📥 Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Keep hierarchy names and codes (drop Ids — use names for BI display)
- Drop: SecurityCode, all ETL dates
- Keep GlobalFinancialSegmentId as PK

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("GlobalFinancialSegmentId").cast("int"),
    F.col("SegmentCode").cast("string"),
    F.col("SegmentName").cast("string"),
    F.col("BusinessCode").cast("string"),
    F.col("BusinessName").cast("string"),
    F.col("LOBCode").cast("string"),
    F.col("LOBName").cast("string"),
    F.col("ProductServiceCode").cast("string"),
    F.col("ProductServiceName").cast("string"),
    F.col("TeamCode").cast("string"),
    F.col("TeamName").cast("string"),
    F.col("IsDeleted").cast("boolean")
)

print(f"✅ After column select: {df_clean.count():,} rows × {len(df_clean.columns)} cols")

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
unknown_row = spark.createDataFrame([(
    -1, "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", False
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"✅ Added Unknown member: {df_final.count():,} rows")

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("GlobalFinancialSegmentId").distinct().count()

print(f"✅ DQ Checks")
print(f"   Total rows:     {total:,}")
print(f"   Duplicate PKs:  {dupes}")
assert dupes == 0, f"❌ Duplicates found!"
print("\n✅ All DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{LAKEHOUSE}.{TABLE}")

print(f"✅ Written: {LAKEHOUSE}.{TABLE}")
print(f"   Rows: {spark.table(f'{LAKEHOUSE}.{TABLE}').count():,}")